<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/learn-ml-algorithms-2017-notebooks/blob/main/10-support-vector-machines.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/learn-ml-algorithms-2017/) collection._

# Chapter 10 — Support Vector Machines

Companion to [the chapter](https://www.ai.biz/books/learn-ml-algorithms-2017/support-vector-machines/).

The widest street between two classes, the handful of points that define it, and the trick that lets a straight line bend.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

def make_two_blobs(n, gap=3.0, std=0.8, seed=0):
    """Two classes of customers in two standardised features. The default gap keeps them separable."""
    rng = np.random.default_rng(seed)
    a = rng.normal([-gap / 2, -gap / 2], std, (n // 2, 2))
    b = rng.normal([gap / 2, gap / 2], std, (n // 2, 2))
    X = np.vstack([a, b])
    y = np.array([-1] * (n // 2) + [1] * (n // 2))        # SVMs like labels of -1 and +1
    return X, y

X, y = make_two_blobs(100, seed=1)
X_test, y_test = make_two_blobs(2000, seed=2)
print(f'{len(y)} points, labels {np.unique(y)}')

## 1. Many lines separate the classes. Which one?

In [ ]:
def margin_of(w, b, X, y):
    """The smallest signed distance from any point to the line w·x + b = 0."""
    return np.min(y * (X @ w + b)) / np.linalg.norm(w)

candidates = {'steep': (np.array([3.0, 1.0]), 0.3), 'diagonal': (np.array([1.0, 1.0]), 0.0), 'flat': (np.array([1.0, 3.0]), -0.2)}
for name, (w, b) in candidates.items():
    train_ok = np.mean(np.sign(X @ w + b) == y)
    test_ok = np.mean(np.sign(X_test @ w + b) == y_test)
    print(f'{name:<9} separates training data: {train_ok:.0%}   margin {margin_of(w, b, X, y):6.3f}   accuracy on new points {test_ok:.3f}')

## 2. Hinge loss: the soft margin as something you can walk down

In [ ]:
def hinge_loss(w, b, X, y, C):
    margins = y * (X @ w + b)
    return 0.5 * w @ w + C * np.sum(np.maximum(0, 1 - margins))

def hinge_gradient(w, b, X, y, C):
    margins = y * (X @ w + b)
    violating = margins < 1                                 # only points inside the street push on the line
    gw = w - C * np.sum((y[violating])[:, None] * X[violating], axis=0)
    gb = -C * np.sum(y[violating])
    return gw, gb

def train_svm(X, y, C, steps=5000, rate=0.001):
    w, b = np.zeros(2), 0.0
    for _ in range(steps):
        gw, gb = hinge_gradient(w, b, X, y, C)
        w, b = w - rate * gw, b - rate * gb
    return w, b

w, b = train_svm(X, y, C=1.0)
print(f'w = {np.round(w, 3)}, b = {b:.3f}')
print(f'margin {margin_of(w, b, X, y):.3f}   (2/|w| = {2 / np.linalg.norm(w):.3f} is the width of the street)')
print(f'accuracy on new points {np.mean(np.sign(X_test @ w + b) == y_test):.3f}')

## 3. The one line you would ship, and a check

In [ ]:
from sklearn.svm import SVC, LinearSVC

svc = SVC(kernel='linear', C=1.0).fit(X, y)
print(f'scikit-learn w = {np.round(svc.coef_[0], 3)}, b = {svc.intercept_[0]:.3f}')
print(f'accuracy on new points {svc.score(X_test, y_test):.3f}')
print(f'support vectors: {len(svc.support_)} of {len(y)} training points')

## 4. Only the support vectors matter

In [ ]:
keep = np.setdiff1d(np.arange(len(y)), svc.support_)      # throw away every point that is NOT a support vector
far_only = SVC(kernel='linear', C=1.0).fit(X[svc.support_], y[svc.support_])
print('refit on the support vectors alone: w =', np.round(far_only.coef_[0], 3), ' b =', round(far_only.intercept_[0], 3))

X_moved = X.copy()
X_moved[keep] += np.abs(rng.normal(0, 0.5, (len(keep), 2))) * np.sign(X_moved[keep])     # push the far points further out
moved = SVC(kernel='linear', C=1.0).fit(X_moved, y)
print('move every non-support point further away: w =', np.round(moved.coef_[0], 3), ' b =', round(moved.intercept_[0], 3))

## 5. C: how much a violation costs

In [ ]:
X_o, y_o = make_two_blobs(100, gap=1.2, std=0.9, seed=3)          # overlapping classes
X_ot, y_ot = make_two_blobs(2000, gap=1.2, std=0.9, seed=4)
print(f'{"C":>7} {"support vectors":>16} {"street width":>13} {"train":>7} {"test":>7}')
for C in [0.01, 0.1, 1, 10, 100]:
    m = SVC(kernel='linear', C=C).fit(X_o, y_o)
    width = 2 / np.linalg.norm(m.coef_[0])
    print(f'{C:>7} {len(m.support_):>16} {width:>13.2f} {m.score(X_o, y_o):>7.3f} {m.score(X_ot, y_ot):>7.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, C in zip(axes, [0.01, 100]):
    m = SVC(kernel='linear', C=C).fit(X_o, y_o)
    ax.scatter(X_o[:, 0], X_o[:, 1], c=y_o, cmap='coolwarm', s=18, edgecolor='k')
    ax.scatter(X_o[m.support_, 0], X_o[m.support_, 1], s=90, facecolors='none', edgecolors='k')
    xx = np.linspace(-3, 3, 50)
    w_, b_ = m.coef_[0], m.intercept_[0]
    for off, ls in [(0, '-'), (1, '--'), (-1, '--')]:
        ax.plot(xx, (off - b_ - w_[0] * xx) / w_[1], 'k', ls=ls, lw=1)
    ax.set_title(f'C = {C}: {len(m.support_)} support vectors'); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
plt.show()

## 6. The kernel trick, with the feature map written out

In [ ]:
def phi(X):
    """Explicit degree-2 polynomial features for two inputs."""
    x1, x2 = X[:, 0], X[:, 1]
    return np.column_stack([np.ones(len(X)), np.sqrt(2) * x1, np.sqrt(2) * x2,
                            x1**2, x2**2, np.sqrt(2) * x1 * x2])

def poly_kernel(A, B):
    return (A @ B.T + 1) ** 2

A = X[:3]
print('dot products in the six-dimensional feature space:\n', np.round(phi(A) @ phi(A).T, 3))
print('the kernel, computed in two dimensions:\n', np.round(poly_kernel(A, A), 3))

## 7. XOR: a problem no line can solve

In [ ]:
rng = np.random.default_rng(5)
Xx = rng.uniform(-1, 1, (300, 2))
yx = np.where(Xx[:, 0] * Xx[:, 1] > 0, 1, -1)              # top-right and bottom-left are one class
Xx_t = rng.uniform(-1, 1, (3000, 2))
yx_t = np.where(Xx_t[:, 0] * Xx_t[:, 1] > 0, 1, -1)

for name, m in [('linear', SVC(kernel='linear')),
                ('polynomial, degree 2', SVC(kernel='poly', degree=2, coef0=1)),
                ('RBF', SVC(kernel='rbf', gamma=1.0))]:
    m.fit(Xx, yx)
    print(f'{name:<22} accuracy {m.score(Xx_t, yx_t):.3f}   support vectors {len(m.support_)}')

## 8. Gamma: how far one point's influence reaches

In [ ]:
from sklearn.datasets import make_moons

Xm, ym = make_moons(200, noise=0.25, random_state=0)
Xm_t, ym_t = make_moons(3000, noise=0.25, random_state=1)
print(f'{"gamma":>6} {"train":>7} {"test":>7} {"support vectors":>16}')
for gamma in [0.01, 0.1, 1, 10, 100]:
    m = SVC(kernel='rbf', gamma=gamma, C=1).fit(Xm, ym)
    print(f'{gamma:>6} {m.score(Xm, ym):>7.3f} {m.score(Xm_t, ym_t):>7.3f} {len(m.support_):>16}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
xx, yy = np.meshgrid(np.linspace(-1.5, 2.5, 200), np.linspace(-1, 1.5, 200))
for ax, gamma in zip(axes, [0.1, 1, 100]):
    m = SVC(kernel='rbf', gamma=gamma, C=1).fit(Xm, ym)
    Z = m.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z > 0, alpha=0.2, cmap='coolwarm')
    ax.scatter(Xm[:, 0], Xm[:, 1], c=ym, cmap='coolwarm', s=12, edgecolor='k', linewidth=0.3)
    ax.set_title(f'gamma = {gamma}')
plt.tight_layout(); plt.show()

## 9. Scale first, always

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

Xs = Xm.copy(); Xs[:, 0] *= 1000                           # the first feature in metres instead of kilometres
Xs_t = Xm_t.copy(); Xs_t[:, 0] *= 1000
print(f'RBF on raw features      : {SVC(gamma="scale").fit(Xs, ym).score(Xs_t, ym_t):.3f}')
print(f'RBF after StandardScaler : {make_pipeline(StandardScaler(), SVC(gamma="scale")).fit(Xs, ym).score(Xs_t, ym_t):.3f}')

## 10. Choosing C and gamma honestly

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = {'svc__C': [0.1, 1, 10, 100], 'svc__gamma': [0.01, 0.1, 1, 10]}
search = GridSearchCV(make_pipeline(StandardScaler(), SVC()), grid, cv=5).fit(Xm, ym)
print(f'best by cross-validation: {search.best_params_}   test accuracy {search.score(Xm_t, ym_t):.3f}')

## 11. The kernel matrix grows with the square of the data

In [ ]:
import time

for n in [1000, 4000, 16000]:
    Xn, yn = make_moons(n, noise=0.25, random_state=0)
    start = time.perf_counter()
    SVC(kernel='rbf', gamma=1).fit(Xn, yn)
    print(f'n = {n:>5}: {time.perf_counter() - start:6.2f} s to train   ({n * n / 1e6:7.1f} million kernel entries)')

## 12. Imbalanced classes

In [ ]:
rng = np.random.default_rng(8)
Xi, yi = make_moons(2000, noise=0.3, random_state=3)
keep = (yi == 0) | (rng.uniform(size=len(yi)) < 0.05)   # keep every majority point and 5% of the minority
Xi, yi = Xi[keep], yi[keep]
Xi_t, yi_t = make_moons(4000, noise=0.3, random_state=4)
print(f'{len(yi)} points, {np.mean(yi == 1):.1%} in the minority class')
for name, m in [('plain', SVC(gamma=1)), ('class_weight=balanced', SVC(gamma=1, class_weight='balanced'))]:
    pred = m.fit(Xi, yi).predict(Xi_t)
    recall = np.mean(pred[yi_t == 1] == 1)
    print(f'{name:<22} accuracy {np.mean(pred == yi_t):.3f}   minority recall {recall:.2f}')

## 13. Support vector regression: the tube

In [ ]:
from sklearn.svm import SVR

def bicycle(km):
    return 10 + 3.2 * km + 0.15 * km**2

rng = np.random.default_rng(1)
km = rng.uniform(0.5, 8.0, 200); minutes = bicycle(km) + rng.normal(0, 4.0, 200)
km_t = rng.uniform(0.5, 8.0, 1000); min_t = bicycle(km_t) + rng.normal(0, 4.0, 1000)
print(f'{"epsilon":>7} {"support vectors":>16} {"MAE on new orders":>18}')
for eps in [0.1, 1, 4, 8]:
    m = make_pipeline(StandardScaler(), SVR(kernel='rbf', C=100, epsilon=eps)).fit(km.reshape(-1, 1), minutes)
    mae = np.mean(np.abs(m.predict(km_t.reshape(-1, 1)) - min_t))
    print(f'{eps:>7} {len(m[-1].support_):>16} {mae:>18.2f}')

## Try it yourself

1. In section 2, change `C` to 100 and to 0.01 and re-run the from-scratch trainer. Which points become support vectors?
2. In section 7, try the polynomial kernel with `degree=3`. Does XOR still work? What about `coef0=0`?
3. In section 11, time `LinearSVC` on the same sizes. Why does its training time grow so much more slowly?